# KTSDE 논문 재실행 — 5시드 × 두 모형 (Kaggle)

논문 〈Table 4〉의 5시드 수치는 근거 파일이 없고, 제안 모형만 GPU·대조군은 MPS 로 학습돼
장비가 섞여 있다. **이 노트북 한 번으로 그 둘을 같이 해결한다.**

| 닫히는 항목 | 어떻게 |
|---|---|
| 〈Table 4〉 출처 없음 | `seed_aggregate.py` 가 리포트를 남긴다 |
| 장비 혼재 | 두 모형을 같은 GPU 에서 5시드씩 학습한다 |
| 5시드 검정구간 미산출 | `--test` 로 검정구간 핀볼까지 낸다 |
| 자료 버전 불명 | `freeze_panel.py --verify` 로 로컬 동결 기록과 대조한다 |

---
### 실행 전 우측 패널에서 3가지

| 설정 | 값 |
|---|---|
| **Accelerator** | `GPU T4 x2` (또는 P100) |
| **Internet** | `On` |
| **Input** → Add Input | 업로드한 `train_bundle.zip` 데이터셋 |

> 회색으로 안 눌리면 **휴대폰 인증**이 필요하다 (Settings → Phone Verification).

### 예상 시간

제안 모형 5시드 약 25분 + TFT 5시드 약 1시간 30분 + 집계 10분 = **약 2시간 5분**.
세션 한도(12시간 / 주당 30시간) 안이다. 셀을 **위에서부터 순서대로** 실행한다.

⚠️ **로컬에서 코드를 고쳤으면 먼저 `git push`** 해야 반영된다. 이 노트북은 GitHub 에서
코드를 새로 클론한다.

## 1. GPU 확인
`cuda: True` 가 나와야 한다. False 면 Accelerator 를 GPU 로 바꾸고 세션 재시작.

In [ ]:
import torch

ok = torch.cuda.is_available()
print('cuda:', ok)
print('GPU :', torch.cuda.get_device_name(0) if ok else '없음')
if not ok:
    raise SystemExit('⚠️ 우측 Accelerator 를 GPU 로 바꾸고 Run → Restart Session')

## 2. 코드 + 데이터 준비

논문 모형은 **기본 트랙**이다. `PROFILE` 은 반드시 `''` 여야 한다 —
태그가 붙으면 `seed_aggregate.py` 가 찾는 이름과 달라지고, 논문 수치와도 다른 트랙이 된다.

In [ ]:
PROFILE = ''          # 논문 트랙. 바꾸지 말 것.
assert PROFILE == '', "이 노트북은 논문 재실행 전용이다 — PROFILE 은 '' 여야 한다"

PROF_ARG = ''
print('학습 트랙: (기본)')

In [ ]:
import glob, os, shutil, zipfile, subprocess

# --- 코드: 항상 새로 받는다 (예전 버전이 남아 헷갈리는 일을 막는다)
os.chdir('/kaggle/working')
shutil.rmtree('Capstone_Stock_Price_Prediction', ignore_errors=True)
subprocess.run(['git', 'clone', '-q',
                'https://github.com/sungjunpk/Capstone_Stock_Price_Prediction.git'],
               check=True)
os.chdir('/kaggle/working/Capstone_Stock_Price_Prediction')
print('최신 코드:', subprocess.run(['git', 'log', '-1', '--format=%h %s'],
                                capture_output=True, text=True).stdout.strip())

# --- 데이터: Input 에서 zip 이든 parquet 이든 찾아서 푼다
os.makedirs('data/processed', exist_ok=True)
zips  = glob.glob('/kaggle/input/**/*.zip', recursive=True)
parqs = glob.glob('/kaggle/input/**/panel*.parquet', recursive=True)

if parqs:
    src = os.path.dirname(parqs[0])
    for f in glob.glob(f'{src}/*.parquet'):
        shutil.copy(f, 'data/processed/')
    print('데이터:', src)
elif zips:
    with zipfile.ZipFile(zips[0]) as z:
        z.extractall('.')
    print('데이터:', zips[0])
else:
    raise SystemExit('❌ 데이터를 못 찾았다 — 우측 Add Input 을 확인할 것')

print('준비된 파일:', sorted(os.listdir('data/processed')))
assert os.path.exists('data/processed/panel.parquet'), (
    '❌ panel.parquet 이 없다. 접미사 붙은 묶음(panel_xs.parquet 등)을 올린 것 같다 — '
    '`python scripts/package_data.py` (프로파일 없이) 로 만든 zip 을 쓸 것.')

# --- 설정 확인: 낡은 코드로 도는 사고를 여기서 잡는다
import yaml
enc = yaml.safe_load(open('configs/config.yaml'))['model']['encoder']
print(f"\n모델 설정: d_model={enc['d_model']} n_layers={enc['n_layers']} dropout={enc['dropout']}")
assert enc['d_model'] == 32 and enc['n_layers'] == 1, (
    '❌ 옛날 설정이다. 학습 로그에 파라미터가 0.50M 이 아니면 이 셀부터 다시 실행할 것.')
print('✅ 확정 설정 맞음 (파라미터 0.50M 이 나와야 한다)')

## 3. 자료 동결 대조 ★

로컬에서 동결한 기록과 **윈도우 수·행 수·기간·피처 수**가 같은지 본다.
여기서 멈추면 데이터셋 버전이 다른 것이다 — 아래를 다 돌려도 논문과 묶이지 않는다.

> sha256 은 다르게 나오는 게 정상이다. `package_data.py` 가 학습에 안 쓰는 열을 떼고
> float32 로 다시 쓰기 때문이다.

❗ **캐글 함정**: 데이터셋에 New Version 을 올려도 노트북 Input 은 **옛 버전에 고정**된다.
어긋나면 우측 Input 에서 데이터셋을 **제거하고 다시 추가**할 것.

In [ ]:
!python scripts/freeze_panel.py --verify docs/freeze_panel_20261018.json

## 4. 배관 점검 (1~2분)

전체 학습 전에 소규모로 한 번 돌린다. 여기서 에러가 나면 아래도 다 실패한다.

In [ ]:
!python scripts/train.py --smoke

## 5. 제안 모형 5시드 (약 25분)

`--seed` 를 주면 체크포인트에 `_seedN` 태그가 자동으로 붙는다. 태그가 없으면
① 시드끼리 서로를 덮어쓰고 ② 무태그라 `paper_trade.py` 의 실주문 경로가 집어간다.

**batch-size 를 넘기지 않는다** — config 기본값으로 다섯 시드를 동일하게 둔다.

In [ ]:
import subprocess, time

for s in [42, 43, 44, 45, 46]:
    t0 = time.time()
    print(f"\n===== ours seed {s} " + "=" * 46, flush=True)
    subprocess.run(f"python scripts/train.py --seed {s}", shell=True, check=True)
    print(f"----- ours seed {s}: {(time.time() - t0) / 60:.1f}분", flush=True)

## 6. TFT 원본 5시드 (약 1시간 30분)

`train.py` 에는 `--arch` 가 없다. 아키텍처는 `arch_compare.py` 가 `model.arch` 와
손실을 바꿔 학습한다. `--seed` 는 여기에도 있다.

> 시간이 부족하면 시드를 3개로 줄여도 된다. 다만 그 경우 〈Table 4〉의 TFT 쪽
> 표준편차가 3시드 값이라고 본문에 밝혀야 한다.

In [ ]:
import subprocess, time

for s in [42, 43, 44, 45, 46]:
    t0 = time.time()
    print(f"\n===== tft seed {s} " + "=" * 48, flush=True)
    subprocess.run(f"python scripts/arch_compare.py --train tft --seed {s}",
                   shell=True, check=True)
    print(f"----- tft seed {s}: {(time.time() - t0) / 60:.1f}분", flush=True)

## 7. 〈Table 4〉 집계 ★ (5분)

시드별 체크포인트에서 검증·검정 두 구간의 핀볼 손실과 같은 표본의 상수 기준선을
다시 계산하고, 시드를 짝지어 부호검정(단측)·Welch 검정(양측)을 낸다.
**이 셀이 만드는 `seed_aggregate_*.json` 이 〈Table 4〉의 출처 파일이다.**

체크포인트 이름 앞부분은 하드코딩하지 않고 실제 파일에서 찾는다 — config 해시가
바뀌어도 돈다.

In [ ]:
import glob, os, re, subprocess

def stem_of(pattern, exclude=()):
    for p in sorted(glob.glob(f'outputs/checkpoints/{pattern}')):
        n = os.path.basename(p)
        if 'smoke' in n or any(e in n for e in exclude):
            continue
        return re.sub(r'_seed\d+\.pt$', '', n)
    raise SystemExit(f'❌ {pattern} 에 맞는 체크포인트가 없다 — 위 학습 셀이 끝났는지 확인')

A = stem_of('phase1_*_seed42.pt', exclude=('_tft',))      # 제안 모형
B = stem_of('phase1_*_tft_seed42.pt')                      # TFT 원본
print('제안 모형:', A)
print('TFT 원본 :', B)

subprocess.run(
    f"python scripts/seed_aggregate.py --a {A} --b {B} "
    f"--seeds 42 43 44 45 46 --test "
    f"--freeze docs/freeze_panel_20261018.json",
    shell=True, check=True)

## 8. 시드별 DM 검정 (5분)

논문 5.1절의 검정구간 비교다. 같은 표본 위의 날짜별 손실 계열 차를 보는 검정이라
**재학습이 필요 없다**. 5시드 모두 같은 장비 체크포인트로 돌리므로 장비 혼재가 사라진다.

> `dm_test.py` 의 `pinball()` 은 세 분위의 **합**이다. 본문 수치는 분위 평균이므로
> 손실값을 쓸 때는 3으로 나눌 것(DM 통계량은 척도 불변이라 그대로 쓴다).

In [ ]:
import subprocess

for s in [42, 43, 44, 45, 46]:
    print(f"\n===== DM seed {s} " + "=" * 48, flush=True)
    subprocess.run(
        f"python scripts/dm_test.py "
        f"--a outputs/checkpoints/{A}_seed{s}.pt "
        f"--b outputs/checkpoints/{B}_seed{s}.pt",
        shell=True, check=True)

## 9. 결과 내려받기

우측 **Output** 패널에 `ktsde_rerun.zip` 이 생긴다. 받아서 로컬 저장소의 `outputs/` 에 푼다.

로컬에서 할 나머지(추론만, GPU 불필요): 〈Table 6〉 기권 지표, 〈Table 7〉 구간 교정,
기권율 맞춤 비교를 **새 체크포인트로** 다시 산출한다.

In [ ]:
import os, shutil

shutil.rmtree('/kaggle/working/download', ignore_errors=True)
os.makedirs('/kaggle/working/download', exist_ok=True)
for p in ['outputs/checkpoints', 'outputs/reports']:
    if os.path.exists(p):
        shutil.copytree(p, f'/kaggle/working/download/{os.path.basename(p)}')

shutil.make_archive('/kaggle/working/ktsde_rerun', 'zip', '/kaggle/working/download')
print('→ 우측 Output 패널에서 ktsde_rerun.zip 다운로드')
!ls -lh /kaggle/working/ktsde_rerun.zip

---
## 문제 해결

| 증상 | 해결 |
|---|---|
| `cuda: False` | Accelerator 를 GPU 로 → Run → Restart Session |
| `자료가 다르다` (3번 셀) | Input 데이터셋이 옛 버전이다. 제거 후 다시 추가 |
| `panel.parquet 이 없다` | 프로파일 붙은 묶음을 올렸다. `package_data.py` 를 인자 없이 다시 |
| `데이터를 못 찾았다` | 우측 **Add Input** 으로 데이터셋을 붙인다 |
| `git clone` 이 멈춤 | **Internet: On** 확인 |
| 파라미터가 0.50M 이 아니다 | 낡은 클론이다. 2번 셀부터 다시 |
| `CUDA out of memory` | 학습 셀에 `--batch-size 128` 을 붙인다 |
| 세션이 끊김 | `/kaggle/working` 이 비므로 **2번 셀부터** 다시 |

## 이 노트북을 다 돌린 뒤 바뀌는 것

〈Table 4〉의 다섯 시드 수치가 새 값으로 바뀐다. 검정구간 개선율은 검증구간보다
훨씬 크게 나온다(로컬 1시드에서 +4.8% 대 +2.1%) — 검정구간 기준선이 더 커서 그렇다.
**본문에 넣을 때 어느 구간 값인지 반드시 밝혀야 한다.**